***Create the FASTA input file***

for learning: in data/sequences.fasta and paste this:

>native
KVFGRCELAAAMKRHGLDNYRGYSLGNWVCAAKFESNFNTQATNRNTDGSTDYGILQINSRWWCNDGRTPGSRNLCNIPCSALLSSDITASVNCAKKIVSDGNGMNAWVAWRNRCKGTDVQAWIRGCRL
>design_1
KVFGRCELAAAMKRHGLDNYRGYSLGNWVCAAKFESNFNTQATNRNTDGSTDYGILQINSRWWCNDGRTPGSRNLCNIPCSALLSSDVTASVNCAKKIVSDGNGMNAWVAWRNRCKGTDVQAWIRGCRL
>design_2
KVFGRCELAAAMKRHGLDNYRGYSLGNWVCAAKFESNFNTQATNRNTDGSTDYGILQINSRWWCNDGRTPGSRNLCNIPCSALLSSDITASVNCAKKIVSDGNGMNAWVAWRNRCKGTDVQAWIRGCRF
>design_3
KVFGRCELAAAMKRHGLDNYRGYSLGNWVCAAKFESNFNTQATNRNTDGSTDYGILQINSRWWCNDGRTPGSRNLCNIPCSALLSSDITASVNCAKKIVSDGNGMNAWVAWRNRCKGTDVQAWIRGCKL

# Protein Design Evaluation Using ESM

This notebook evaluates native and designed protein sequences using ESM2 embeddings and cosine similarity.

In [1]:
from pathlib import Path
from typing import List, Tuple

import esm
import pandas as pd
import torch
from Bio import SeqIO
from sklearn.metrics.pairwise import cosine_similarity

***File paths***

In [2]:
input_path = Path("../data/sequences.fasta")
results_dir = Path("../results")
results_dir.mkdir(parents=True, exist_ok=True)

input_path

PosixPath('../data/sequences.fasta')

***Load FASTA function***

In [3]:
def load_fasta_sequences(fasta_path: str | Path) -> List[Tuple[str, str]]:
    fasta_path = Path(fasta_path)

    if not fasta_path.exists():
        raise FileNotFoundError(f"FASTA file not found: {fasta_path}")

    sequences = []
    for record in SeqIO.parse(str(fasta_path), "fasta"):
        seq_name = record.id.strip()
        seq = str(record.seq).strip().upper()

        if not seq:
            raise ValueError(f"Empty sequence found for record: {seq_name}")

        sequences.append((seq_name, seq))

    if not sequences:
        raise ValueError(f"No sequences found in FASTA file: {fasta_path}")

    return sequences

***Equal length check***

In [4]:
def validate_equal_lengths(sequences: List[Tuple[str, str]]) -> None:
    lengths = {len(seq) for _, seq in sequences}
    if len(lengths) != 1:
        raise ValueError(
            f"Sequences do not all have the same length. Found lengths: {sorted(lengths)}"
        )

***Load and inspect sequences***

In [5]:
sequences = load_fasta_sequences(input_path)
validate_equal_lengths(sequences)

for name, seq in sequences:
    print(f"{name:>10} | length = {len(seq)}")

    native | length = 129
  design_1 | length = 129
  design_2 | length = 129
  design_3 | length = 129


***Show sequences as a table***

In [8]:
seq_df = pd.DataFrame(sequences, columns=["name", "sequence"])
seq_df["length"] = seq_df["sequence"].str.len()
seq_df

,name,sequence,length
0,native,KVFGRCELAAAMKRHGLDNYRGYSLGNWVCAAKFESNFNTQATNRN...,129
1,design_1,KVFGRCELAAAMKRHGLDNYRGYSLGNWVCAAKFESNFNTQATNRN...,129
2,design_2,KVFGRCELAAAMKRHGLDNYRGYSLGNWVCAAKFESNFNTQATNRN...,129
3,design_3,KVFGRCELAAAMKRHGLDNYRGYSLGNWVCAAKFESNFNTQATNRN...,129


***Load ESM model***

In [9]:
model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
batch_converter = alphabet.get_batch_converter()
model.eval()

print("ESM model loaded successfully")

Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t6_8M_UR50D.pt" to /Users/vaibhav.mh/.cache/torch/hub/checkpoints/esm2_t6_8M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t6_8M_UR50D-contact-regression.pt" to /Users/vaibhav.mh/.cache/torch/hub/checkpoints/esm2_t6_8M_UR50D-contact-regression.pt
ESM model loaded successfully


***Compute embeddings function***

In [10]:
def compute_esm_embeddings(sequences: List[Tuple[str, str]]) -> pd.DataFrame:
    batch_labels, batch_strs, batch_tokens = batch_converter(sequences)

    with torch.no_grad():
        results = model(batch_tokens, repr_layers=[6], return_contacts=False)

    token_representations = results["representations"][6]

    rows = []
    for i, (label, seq) in enumerate(sequences):
        seq_len = len(seq)

        # Remove special tokens and average across residues
        seq_embedding = token_representations[i, 1:seq_len + 1].mean(0).cpu().numpy()

        row = {"name": label}
        for j, value in enumerate(seq_embedding):
            row[f"emb_{j}"] = float(value)

        rows.append(row)

    return pd.DataFrame(rows)

***Run embedding computation***

In [12]:
embedding_df = compute_esm_embeddings(sequences)
embedding_df.iloc[:, :8]

,name,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6
0,native,0.054077,-0.006217,0.093555,0.098339,0.081594,-0.114220,-0.160251
1,design_1,0.048080,0.001278,0.103753,0.101737,0.076322,-0.119807,-0.163014
2,design_2,0.068742,0.007372,0.096252,0.106946,0.093113,-0.115592,-0.164522
3,design_3,0.054096,0.007211,0.096675,0.103272,0.084940,-0.095066,-0.166718


***save embeddings***

In [21]:
embedding_path = results_dir / "embeddings.csv"
embedding_df.to_csv(embedding_path, index=False)

print(f"Saved embeddings to: {embedding_path}")

Saved embeddings to: ../results/embeddings.csv


***Similarity matrix function***

In [15]:
def compute_similarity_matrix(embedding_df: pd.DataFrame) -> pd.DataFrame:
    if "name" not in embedding_df.columns:
        raise ValueError("Input embedding DataFrame must contain a 'name' column.")

    names = embedding_df["name"].tolist()
    X = embedding_df.drop(columns=["name"]).values

    sim = cosine_similarity(X)
    sim_df = pd.DataFrame(sim, index=names, columns=names)
    return sim_df

***Ranking function***

In [16]:
def rank_designs_against_native(sim_df: pd.DataFrame, native_name: str = "native") -> pd.DataFrame:
    if native_name not in sim_df.columns:
        raise ValueError(f"Native sequence '{native_name}' not found in similarity matrix.")

    ranking_df = sim_df[[native_name]].reset_index()
    ranking_df.columns = ["name", "similarity_to_native"]
    ranking_df = ranking_df[ranking_df["name"] != native_name].copy()
    ranking_df = ranking_df.sort_values(
        by="similarity_to_native",
        ascending=False
    ).reset_index(drop=True)

    ranking_df["rank"] = range(1, len(ranking_df) + 1)
    ranking_df = ranking_df[["rank", "name", "similarity_to_native"]]
    return ranking_df

***Compute similarity matrix***

In [17]:
sim_df = compute_similarity_matrix(embedding_df)
sim_df.round(4)

,native,design_1,design_2,design_3
native,1.0000,0.9998,0.9997,0.9994
design_1,0.9998,1.0000,0.9995,0.9992
design_2,0.9997,0.9995,1.0000,0.9992
design_3,0.9994,0.9992,0.9992,1.0000


***Rank designs***

In [18]:
ranking_df = rank_designs_against_native(sim_df, native_name="native")
ranking_df.round(4)

,rank,name,similarity_to_native
0,1,design_1,0.9998
1,2,design_2,0.9997
2,3,design_3,0.9994


***Save outputs***

In [19]:
sim_path = results_dir / "similarity_matrix.csv"
rank_path = results_dir / "design_ranking.csv"

sim_df.to_csv(sim_path)
ranking_df.to_csv(rank_path, index=False)

print(f"Saved similarity matrix to: {sim_path}")
print(f"Saved design ranking to: {rank_path}")

Saved similarity matrix to: ../results/similarity_matrix.csv
Saved design ranking to: ../results/design_ranking.csv


***Simple interpretation***

In [20]:
best_design = ranking_df.iloc[0]
print("Best design based on ESM similarity:")
print(best_design)

Best design based on ESM similarity:
rank                           1
name                    design_1
similarity_to_native    0.999827
Name: 0, dtype: object
